# Request-time and customer-profile feature contracts

The two models have different populations and different meanings. Applicant scoring must never substitute a current profile for historical request inputs. Customer scoring uses the supplied actual profile and never constructs a fictitious loan.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
OUT = ROOT / 'artifacts'
assert json.loads((OUT / 'run_manifest.json').read_text())['version'] == 2, 'Run python -m src.pipeline first'

from src.data_load import read_raw
from src.features import strict_application_features, customer_profile_features, STRICT_FEATURES, PROFILE_FEATURES
frames=read_raw()
requests=strict_application_features(frames)
profiles=customer_profile_features(frames)
print('Requests:',requests.shape,'Customer profiles:',profiles.shape)
assert requests.application_id.is_unique and profiles.customer_id.is_unique

## Strict request-time construction
Use request amount, term, type, purpose, collateral, declared monthly income and existing EMI. Build existing-payment burden, request-to-income and principal-only instalment burden without inventing an interest rate. Earlier transaction windows are six calendar months and exclude the application day. Undated demographics, bureau values, offered rates and supplied underwriting DTI are excluded entirely.

In [ ]:
blocked={'credit_score','annual_income','employment_type','interest_rate','debt_to_income_ratio','approval_status','approved_amount','decision_date','disbursed_flag','loan_default'}
assert not blocked.intersection(STRICT_FEATURES)
display(requests[STRICT_FEATURES].isna().mean().sort_values(ascending=False).to_frame('missing_fraction'))
print('History available:',requests.history_available.value_counts().to_dict())

## Verify future transaction isolation
Appending a large future transaction must not alter historical feature vectors. This is an implementation property independent of reported test metrics.

In [ ]:
sample=frames['loan_applications'].head(30)
source={'loan_applications':sample,'transactions':frames['transactions']}
before=strict_application_features(source)
future=frames['transactions'].head(20).copy();future['transaction_date']='2030-01-01';future['amount']=1e12
source['transactions']=pd.concat([frames['transactions'],future],ignore_index=True)
after=strict_application_features(source)
pd.testing.assert_frame_equal(before,after)
print('Future transaction isolation passed.')

## Actual customer profiles
Every customer uses their own bureau, customer and account attributes. No loan amount, tenure or product is invented. The model is retrospective: its training target is the latest observed disbursed loan label per customer, and the bureau snapshot may be after that outcome.

In [ ]:
assert not {'requested_amount','loan_type','loan_term_months','loan_to_income'}.intersection(PROFILE_FEATURES)
display(profiles.drop(columns='customer_id').describe().T)
print('Customer-only features:',PROFILE_FEATURES)

## Fold-local preprocessing
Numeric medians and Unknown/one-hot categorical encoding are learned inside each training fold. Only logistic regression standardises numeric features. Missing history is distinct from zero observed cashflow. The persisted artefact fixes the exact model input contract.

In [ ]:
import joblib
artifact=joblib.load(ROOT/'models/default_model.pkl')
assert artifact['features']==STRICT_FEATURES
print('Applicant contract:',artifact['features'])
print((ROOT/'documentation/feature_contracts.md').read_text())